# Прогноз объема обращений

Ноутбук показывает постановку задачи, признаки, baseline и интерпретацию прогноза для планирования смен.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

root = Path('..')
processed = root / 'data' / 'processed'
marts = root / 'data' / 'marts'

## Подготовка дневного ряда

Агрегируем тикеты по дате создания и добавляем календарные признаки.

In [ ]:
tickets = pd.read_csv(processed / 'tickets.csv', parse_dates=['created_at'])
daily = tickets.assign(date=tickets['created_at'].dt.date).groupby('date').size().rename('actual_tickets').reset_index()
daily['date'] = pd.to_datetime(daily['date'])
daily['day_of_week'] = daily['date'].dt.dayofweek + 1
daily['month'] = daily['date'].dt.month
daily['is_weekend'] = (daily['day_of_week'] >= 6).astype(int)
daily.head()

In [ ]:
daily['lag_1'] = daily['actual_tickets'].shift(1)
daily['lag_7'] = daily['actual_tickets'].shift(7)
daily['rolling_7_mean'] = daily['actual_tickets'].shift(1).rolling(7).mean()
model_data = daily.dropna().copy()
model_data.tail()

## Baseline и модель

Сравниваем модель со скользящим средним за семь дней.

In [ ]:
test_days = 28
features = ['day_of_week', 'month', 'is_weekend', 'lag_1', 'lag_7', 'rolling_7_mean']
train = model_data.iloc[:-test_days]
test = model_data.iloc[-test_days:]

model = RandomForestRegressor(n_estimators=150, random_state=42, max_depth=6)
model.fit(train[features], train['actual_tickets'])
test = test.copy()
test['model_forecast'] = model.predict(test[features])
test['moving_average_forecast'] = test['rolling_7_mean']

mae_model = mean_absolute_error(test['actual_tickets'], test['model_forecast'])
mape_model = mean_absolute_percentage_error(test['actual_tickets'], test['model_forecast']) * 100
mae_baseline = mean_absolute_error(test['actual_tickets'], test['moving_average_forecast'])
mape_baseline = mean_absolute_percentage_error(test['actual_tickets'], test['moving_average_forecast']) * 100
mae_model, mape_model, mae_baseline, mape_baseline

## Визуализация

Смотрим факт, прогноз модели и baseline на тестовом периоде.

In [ ]:
plot_data = test.set_index('date')[['actual_tickets', 'model_forecast', 'moving_average_forecast']]
plot_data.plot(figsize=(12, 4))
plt.title('факт и прогноз объема обращений')
plt.ylabel('tickets')
plt.show()

## Бизнес-интерпретация

Прогноз помогает заранее планировать смены первой линии, проверять риск роста backlog и готовить резерв на дни после инцидентов или релизов.

## Ограничения

Модель не знает о релизах, маркетинговых кампаниях, праздниках и внешних инцидентах. для промышленного прогноза эти факторы нужно добавить отдельными признаками.

In [ ]:
forecast_mart = pd.read_csv(marts / 'mart_forecast.csv')
forecast_mart.tail(20)